# Defining GPT Model Hyperparameters with GPTConfig

Welcome to the first step in building our own GPT model from scratch! Before we can start assembling the complex machinery of a Transformer, we need a blueprint. By the end of this notebook, you will understand how a simple configuration object acts as this essential blueprint, dictating the size, shape, and power of our future model.

In [ ]:
# === Setup Cell ===
# This notebook uses dataclasses to cleanly store our configuration.
# We'll use matplotlib to create a visual representation of the config.
from dataclasses import dataclass
import matplotlib.pyplot as plt

### The Core Idea: A Blueprint for a Skyscraper

Imagine you're an architect designing a skyscraper. You wouldn't just start welding steel beams together. First, you'd create a detailed blueprint. This document would specify all the critical parameters:
- How many floors will it have? (`n_layer`)
- How wide is the building's main structure? (`n_embd`)
- How many elevator shafts will run through it? (`n_head`)
- What's the maximum occupancy per floor? (`block_size`)

This blueprint doesn't *contain* any steel or glass, but it governs every aspect of the final construction. It allows you to plan, estimate costs (computational resources), and communicate the design to the construction crew (the model-building code).

In the world of nanoGPT, `GPTConfig` is this architectural blueprint. It's a simple container object that holds all the hyperparameters defining our model's structure. By changing the values in this one object, we can create anything from a tiny, two-story "language model" to a massive GPT-2-sized skyscraper, all without changing the construction code itself. This separation of *configuration* from *implementation* is a cornerstone of clean and flexible software design.

In [ ]:
@dataclass
class GPTConfig:
    """The blueprint for our GPT model."""
    block_size: int = 256     # Maximum context length (sequence length)
    vocab_size: int = 1000    # Vocabulary size (number of possible tokens)
    n_layer: int = 4          # Number of Transformer blocks (depth)
    n_head: int = 4           # Number of attention heads (in each block)
    n_embd: int = 64          # Embedding dimension (width of the model)

### Walking Through the Implementation

Let's break down our simple blueprint, line by line.

- **`@dataclass`**: This is a Python "decorator" that automatically adds useful methods to our class. Think of it as a helpful assistant that writes the boilerplate `__init__` constructor for us, so we can just focus on defining the fields we need. It also gives us a nice, readable way to print the object later.

- **`block_size: int = 256`**: This is the model's "context window." It determines the maximum number of tokens the model can look at simultaneously when making a prediction. A `block_size` of 256 means the model can read up to 256 tokens of history to predict the 257th token.

- **`vocab_size: int = 1000`**: This is the size of our model's dictionary. It defines how many unique words or sub-word tokens the model knows. For every token, the model needs to be able to output a probability, so the final output layer will have `vocab_size` dimensions.

- **`n_layer: int = 4`**: This defines the *depth* of the model. A Transformer is built by stacking identical blocks on top of each other. More layers allow the model to learn more complex patterns and abstractions, but also make it slower and more expensive to train.

- **`n_head: int = 4`**: Inside each Transformer block is the attention mechanism, which can be split into multiple "heads." Think of each head as giving the model a different perspective on the input text. One head might focus on grammatical relationships, while another focuses on semantic meaning. `n_head` specifies how many of these parallel perspectives we want.

- **`n_embd: int = 64`**: This is the embedding dimension, which you can think of as the *width* of the model. It's the size of the vector used to represent each token as it flows through the model's layers. A larger `n_embd` allows for more nuanced and detailed representations, but again, increases the model's size and computational cost.

In [ ]:
# --- Demonstration: Creating Different Model Blueprints ---

# Let's define a configuration for a very small "toy" model.
# We'll call it GPT-Nano.
gpt_nano_config = GPTConfig(
    block_size=64,
    vocab_size=500,
    n_layer=3,
    n_head=3,
    n_embd=48,
)

print("--- GPT-Nano Config ---")
print(gpt_nano_config)
print("\n" + "="*30 + "\n")

# Now, let's define a configuration that mimics the original "GPT-2 small" (124M params).
# Note: The real vocab_size is 50257, but we simplify.
gpt2_small_config = GPTConfig(
    block_size=1024,
    vocab_size=50257,
    n_layer=12,
    n_head=12,
    n_embd=768,
)

print("--- GPT-2 Small-like Config ---")
print(gpt2_small_config)

### Going Deeper: Why the Funny `vocab_size`?

In the original nanoGPT source code (and in the real GPT-2), you'll notice a peculiar `vocab_size`: 50304. This seems oddly specific. The actual number of tokens in the GPT-2 vocabulary is 50257. So why the increase?

The answer lies in hardware efficiency. Modern GPUs, especially those with Tensor Cores used for deep learning, perform matrix multiplications much, much faster when the dimensions of the matrices are multiples of a certain number, often 8 or even 64.

The final layer of a GPT model is a large linear layer that maps the internal model dimension (`n_embd`) to the vocabulary size (`vocab_size`). Its weight matrix has a shape of `(n_embd, vocab_size)`. By padding the `vocab_size` from 50257 up to the next multiple of 64, which is 50304, the developers ensure that this final, critical matrix multiplication can leverage the GPU's most efficient computation paths. This small, seemingly arbitrary change can lead to a noticeable speedup in training, saving both time and money. It's a perfect example of how low-level hardware considerations can influence high-level model architecture design.

In [ ]:
import matplotlib.pyplot as plt
from dataclasses import asdict

# Configuration for our models
gpt_nano_config = GPTConfig(block_size=64, vocab_size=500, n_layer=3, n_head=3, n_embd=48)
gpt2_small_config = GPTConfig(block_size=1024, vocab_size=50257, n_layer=12, n_head=12, n_embd=768)

# Function to create a table visualization
def visualize_config_comparison(config1, config2, title1="GPT-Nano", title2="GPT-2 Small"):
    """Creates a matplotlib table to compare two GPTConfig objects."""
    
    # Extract data from dataclasses
    data1 = asdict(config1)
    data2 = asdict(config2)
    
    # Prepare table data
    param_names = list(data1.keys())
    cell_text = [
        [f"{data1[key]:,}" for key in param_names],
        [f"{data2[key]:,}" for key in param_names]
    ]
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.axis('tight')
    ax.axis('off')

    # Create the table
    table = ax.table(
        cellText=cell_text,
        rowLabels=[title1, title2],
        colLabels=[name.replace('_', ' ').title() for name in param_names],
        loc='center',
        cellLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(12)
    table.scale(1.2, 1.5)

    # Style the table
    for (i, j), cell in table.get_celld().items():
        if i == 0: # Header row
            cell.set_text_props(weight='bold')
        if i > 0 and j == -1: # Row labels
             cell.set_text_props(weight='bold')

    fig.suptitle("GPT Model Blueprint Comparison", fontsize=16)
    plt.show()

# Visualize the comparison
visualize_config_comparison(gpt_nano_config, gpt2_small_config)

### Summary & What's Next

In this notebook, we established the fundamental concept of using a configuration object as a blueprint for a neural network.

**What we built:**
- A simple `GPTConfig` dataclass to hold the key architectural parameters of a GPT model.
- We saw how this single class can define models of vastly different scales, from a tiny "nano" version to one that mirrors GPT-2.

**Key Takeaways:**
- **Configuration as Code:** Separating the model's blueprint (`GPTConfig`) from its construction logic is a powerful and flexible design pattern.
- **Hyperparameters Dictate Size:** The values for `n_layer`, `n_head`, and `n_embd` are the primary levers that control the model's parameter count and computational complexity.
- **Design Meets Hardware:** Architectural choices, like padding the `vocab_size`, are often driven by the goal of maximizing performance on specific hardware like GPUs.

Now that we have our blueprint, we're ready to lay the foundation. In the next notebook, we will implement the most important component of the Transformer architecture: **Causal Self-Attention**. This is the mechanism that allows the model to weigh the importance of different words in the context and truly "understand" the input sequence.